In [3]:
import pandas as pd

# Load the dataset
file_path = r'C:\Users\Admin\Desktop\practice\data_project.csv'
data = pd.read_csv(file_path)

In [4]:
# Convert datetime columns to proper datetime format if they exist in the dataset
if 'tpep_pickup_datetime' in data.columns and 'tpep_dropoff_datetime' in data.columns:
    data['tpep_pickup_datetime'] = pd.to_datetime(data['tpep_pickup_datetime'])
    data['tpep_dropoff_datetime'] = pd.to_datetime(data['tpep_dropoff_datetime'])

In [5]:
# QN 1: Extract all trips with trip_distance larger than 50
trips_large_distance = data[data['trip_distance'] > 50]
print(trips_large_distance.head())

       VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
23842         2  2020-01-01 01:53:00   2020-01-01 03:54:00                1   
39013         2  2020-01-01 02:05:00   2020-01-01 03:03:00                1   
41620         1  2020-01-01 03:05:00   2020-01-01 04:16:00                1   
58262         2  2020-01-01 05:36:00   2020-01-01 06:40:00                1   
63024         2  2020-01-01 07:40:00   2020-01-01 08:40:00                1   

       trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
23842          52.30           5                  N           262   
39013          51.23           5                  N           264   
41620          53.80           5                  N           132   
58262          55.23           5                  N           132   
63024          54.19           5                  N           132   

       DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  \
23842           265             

In [6]:
# QN 2: Extract all trips where payment_type is missing
trips_missing_payment = data[data['payment_type'].isnull()]
print(trips_missing_payment)

Empty DataFrame
Columns: [VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge]
Index: []


In [23]:
# QN 3: For each (PULocationID, DOLocationID) pair, determine the number of trips
trips_per_location_pair = data.groupby(['PULocationID', 'DOLocationID']).size().reset_index(name='trip_count')
print(trips_per_location_pair.head())

   PULocationID  DOLocationID  trip_count
0             1             1         163
1             1            50           1
2             1           264          38
3             1           265           2
4             2            95           1


In [8]:
# QN 4: Save rows with missing VendorID, passenger_count, store_and_fwd_flag, payment_type
columns_to_check = ['VendorID', 'passenger_count', 'store_and_fwd_flag', 'payment_type']
bad_rows = data[data[columns_to_check].isnull().any(axis=1)]
bad_rows.to_csv('bad_rows.csv', index=False)  # Save bad rows to a new file
data_cleaned = data.drop(bad_rows.index)      # Remove bad rows from the original dataframe

In [9]:
print(bad_rows)

Empty DataFrame
Columns: [VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge]
Index: []


In [10]:
print(data_cleaned)

         VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0               1  2020-01-01 00:28:00   2020-01-01 00:33:00                1   
1               1  2020-01-01 00:35:00   2020-01-01 00:43:00                1   
2               1  2020-01-01 00:47:00   2020-01-01 00:53:00                1   
3               1  2020-01-01 00:55:00   2020-01-01 01:00:00                1   
4               2  2020-01-01 00:01:00   2020-01-01 00:04:00                1   
...           ...                  ...                   ...              ...   
1048570         1  2020-01-07 07:33:00   2020-01-07 07:38:00                1   
1048571         1  2020-01-07 07:40:00   2020-01-07 07:50:00                1   
1048572         1  2020-01-07 07:19:00   2020-01-07 07:28:00                1   
1048573         1  2020-01-07 07:41:00   2020-01-07 07:45:00                3   
1048574         1  2020-01-07 07:39:00   2020-01-07 07:43:00                1   

         trip_distance  Rat

In [11]:
# QN 5: Add a duration column storing how long each trip has taken
data_cleaned['duration'] = (data_cleaned['tpep_dropoff_datetime'] - data_cleaned['tpep_pickup_datetime']).dt.total_seconds() /60
print(data_cleaned[['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'duration']].head())

  tpep_pickup_datetime tpep_dropoff_datetime  duration
0  2020-01-01 00:28:00   2020-01-01 00:33:00       5.0
1  2020-01-01 00:35:00   2020-01-01 00:43:00       8.0
2  2020-01-01 00:47:00   2020-01-01 00:53:00       6.0
3  2020-01-01 00:55:00   2020-01-01 01:00:00       5.0
4  2020-01-01 00:01:00   2020-01-01 00:04:00       3.0


In [12]:

# QN 6: For each pickup location, determine how many trips have started there
trips_per_pickup = data_cleaned['PULocationID'].value_counts().reset_index(name='trip_count')
trips_per_pickup.columns = ['PULocationID', 'trip_count']
print(trips_per_pickup)

     PULocationID  trip_count
0             132       47923
1             237       42010
2             161       40230
3             236       38638
4             186       38565
..            ...         ...
245           111           1
246            99           1
247           187           1
248           206           1
249             2           1

[250 rows x 2 columns]


In [13]:
# QN 6: For each pickup location, determine how many trips have started there
pickup_trip_counts = data_cleaned.groupby('PULocationID').size().reset_index(name='trip_count')
print(pickup_trip_counts)

# Save the cleaned data and results for reference
data_cleaned.to_csv('cleaned_data.csv', index=False)
pickup_trip_counts.to_csv('pickup_trip_counts.csv', index=False)

     PULocationID  trip_count
0               1         204
1               2           1
2               3          15
3               4        2210
4               5           4
..            ...         ...
245           261        6224
246           262       13458
247           263       22075
248           264        6957
249           265         624

[250 rows x 2 columns]


In [14]:
# QN 7: Cluster the pickup time of the day into 30-minute intervals
data_cleaned['pickup_interval'] = data_cleaned['tpep_pickup_datetime'].dt.floor('30min')
print(data_cleaned[['tpep_pickup_datetime', 'pickup_interval']].head())

  tpep_pickup_datetime     pickup_interval
0  2020-01-01 00:28:00 2020-01-01 00:00:00
1  2020-01-01 00:35:00 2020-01-01 00:30:00
2  2020-01-01 00:47:00 2020-01-01 00:30:00
3  2020-01-01 00:55:00 2020-01-01 00:30:00
4  2020-01-01 00:01:00 2020-01-01 00:00:00


In [15]:
# QN 8: For each interval, determine the average number of passengers and the average fare amount
interval_stats = data_cleaned.groupby('pickup_interval').agg({
    'passenger_count': 'mean',
    'fare_amount': 'mean'
}).reset_index()
print(interval_stats.head())

      pickup_interval  passenger_count  fare_amount
0 2003-01-01 00:00:00              1.0          0.0
1 2008-12-31 23:00:00              6.0         52.0
2 2008-12-31 23:30:00              1.0         52.0
3 2009-01-01 00:00:00              5.0         52.0
4 2009-01-01 00:30:00              3.0          8.0


In [16]:
# QN 9: For each payment type and each interval, determine the average fare amount
fare_by_payment_interval = data_cleaned.groupby(['payment_type', 'pickup_interval'])['fare_amount'].mean().reset_index()
print(fare_by_payment_interval.head())

   payment_type     pickup_interval  fare_amount
0             1 2008-12-31 23:30:00        52.00
1             1 2019-12-18 15:00:00         0.01
2             1 2019-12-18 15:30:00         2.50
3             1 2019-12-31 13:30:00        12.00
4             1 2019-12-31 14:00:00        12.75


In [17]:
# QN 10: For each payment type, determine the interval when the average fare amount is maximum
max_fare_interval = fare_by_payment_interval.loc[fare_by_payment_interval.groupby('payment_type')['fare_amount'].idxmax()]
print(max_fare_interval.head())

      payment_type     pickup_interval  fare_amount
0                1 2008-12-31 23:30:00    52.000000
331              2 2008-12-31 23:00:00    52.000000
810              3 2020-01-04 05:30:00    28.500000
1017             4 2020-01-02 05:30:00    36.933333


In [18]:
# QN 11: For each payment type, determine the interval when the tip-to-fare ratio is maximum
data_cleaned['tip_fare_ratio'] = data_cleaned['tip_amount'] / data_cleaned['fare_amount']
max_ratio_interval = data_cleaned.groupby(['payment_type', 'pickup_interval'])['tip_fare_ratio'].mean().reset_index()
max_tip_ratio_interval = max_ratio_interval.loc[max_ratio_interval.groupby('payment_type')['tip_fare_ratio'].idxmax()]
print( max_tip_ratio_interval.head())

      payment_type     pickup_interval  tip_fare_ratio
20               1 2020-01-01 01:30:00             inf
454              2 2020-01-03 07:30:00        0.000348
697              3 2020-01-01 21:00:00        1.066667
1122             4 2020-01-04 10:00:00        0.224490


In [19]:
# QN 12: Find the location with the highest average fare amount
highest_fare_location = data_cleaned.groupby('PULocationID')['fare_amount'].mean().idxmax()
print("\nLocation with the highest average fare amount:")
print(highest_fare_location)


Location with the highest average fare amount:
187


In [20]:
# QN 13: Build a dataframe for each pickup location and its 5 most common destinations
common_destinations = data_cleaned.groupby('PULocationID')['DOLocationID'].value_counts().groupby(level=0).nlargest(5).reset_index(level=0, drop=True).reset_index()
common_destinations.columns = ['PULocationID', 'DOLocationID', 'trip_count']
print(common_destinations)

      PULocationID  DOLocationID  trip_count
0                1             1         163
1                1           264          38
2                1           265           2
3                1            50           1
4                2            95           1
...            ...           ...         ...
1190           265           265         500
1191           265           264          67
1192           265             1          23
1193           265           135           3
1194           265           162           2

[1195 rows x 3 columns]


In [21]:
# QN 14: For each payment type and interval in the common dataframe, determine the average fare amount
common_data = data_cleaned.merge(common_destinations, on=['PULocationID', 'DOLocationID'], how='inner')
common_fare_stats = common_data.groupby(['payment_type', 'pickup_interval'])['fare_amount'].mean().reset_index()
print(common_fare_stats)

      payment_type     pickup_interval  fare_amount
0                1 2019-12-18 15:00:00     0.010000
1                1 2019-12-18 15:30:00     2.500000
2                1 2019-12-31 13:30:00    10.000000
3                1 2019-12-31 14:00:00     6.000000
4                1 2019-12-31 16:30:00     7.000000
...            ...                 ...          ...
1211             4 2020-01-07 05:30:00     5.166667
1212             4 2020-01-07 06:00:00     1.833333
1213             4 2020-01-07 06:30:00    -3.750000
1214             4 2020-01-07 07:00:00    -2.500000
1215             4 2020-01-07 07:30:00     8.500000

[1216 rows x 3 columns]


In [22]:
# QN 15: Compute the difference of the average fare amount with those computed in point 9
common_fare_stats = common_fare_stats.merge(interval_stats, on='pickup_interval', suffixes=('_common', '_overall'))
common_fare_stats['fare_diff'] = common_fare_stats['fare_amount_common'] - common_fare_stats['fare_amount']
print(common_fare_stats)


KeyError: 'fare_amount'

In [ ]:
# QN 16: Compute the ratio of the differences to the overall fare amounts
common_fare_stats['fare_diff_ratio'] = common_fare_stats['fare_diff'] / common_fare_stats['fare_amount']
print(common_fare_stats.head())  # Shows the first 5 rows


In [ ]:
# QN 17: Build chains of trips
# Adding a chain column and identifying chains based on VendorID, locations, and time constraints
data_cleaned = data_cleaned.sort_values(by=['VendorID', 'tpep_pickup_datetime'])
data_cleaned['chain'] = 0
chain_id = 1
for i in range(1, len(data_cleaned)):
    prev_trip = data_cleaned.iloc[i - 1]
    curr_trip = data_cleaned.iloc[i]
    if (
        prev_trip['VendorID'] == curr_trip['VendorID'] and
        prev_trip['DOLocationID'] == curr_trip['PULocationID'] and
        0 < (curr_trip['tpep_pickup_datetime'] - prev_trip['tpep_dropoff_datetime']).total_seconds() <= 120
    ):
        data_cleaned.at[i, 'chain'] = chain_id
    else:
        chain_id += 1
        data_cleaned.at[i, 'chain'] = chain_id
        print(data_cleaned[['VendorID', 'PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'chain']])


In [ ]:
print(data_cleaned)